# 03 — TFNE-Izhikevich hybrid tutorial

This notebook uses the merged `make_cortex_network` builder.  Izhikevich spikes are treated as fast exploratory point-neuron events; a calibration scale converts spike events into amperes before TFNE source projection.


In [1]:
import os, sys, math, json
from pathlib import Path
import numpy as np

SMOKE_MODE = True
SEED = 7
np.random.seed(SEED)

# Allow running from repo root or from the tutorials directory.
ROOT = Path.cwd()
if not (ROOT / "src" / "jbiophysic").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("ROOT", ROOT)
print("SMOKE_MODE", SMOKE_MODE, "SEED", SEED)


ROOT /mnt/data/jbio_improve/jbiophysic-main
SMOKE_MODE True SEED 7


In [2]:
import jax.numpy as jnp
from jbiophysic.networks import make_cortex_network
from jbiophysic.cells.izhikevich import simulate_izhikevich, REGULAR_SPIKING
from jbiophysic.tfne.fields import make_regular_grid
from jbiophysic.tfne.sources import project_sparse_currents, integrate_source


## Build a small cortical volume specification


In [3]:
N = 60 if SMOKE_MODE else 120
net = make_cortex_network(
    XYZ_mm=(0.5, 0.5, 1.5),
    N=N,
    Ls=[0.4, 0.2, 0.4],
    Ld=[[70, 10, 10, 10], [75, 20, 4, 1], [75, 15, 9, 1]],
    model_family="tfne-izhikevich",
    plasticity_coefficient=0.25,
    seed=SEED,
    min_distance_um=5.0,
)
print({"N": net.n_neurons, "counts": net.counts_by_layer_and_type().tolist(), "tfne_keys": sorted(net.tfne)})


{'N': 60, 'counts': [[17, 3, 2, 2], [9, 2, 1, 0], [18, 4, 2, 0]], 'tfne_keys': ['csd_sign_convention', 'izh_current_to_ampere_scale', 'source_positions_m', 'source_radii_m']}


## Spike generator and calibrated source projection


In [4]:
dt_ms = 0.1
T_ms = 100.0 if SMOKE_MODE else 500.0
time = np.arange(int(T_ms/dt_ms)) * dt_ms
I = np.zeros_like(time, dtype=np.float32)
I[(time >= 20) & (time <= 80)] = 9.0
v, u, spikes = simulate_izhikevich(I, params=REGULAR_SPIKING, dt_ms=dt_ms)
spike_count = int(np.asarray(spikes).sum())
scale_A = float(net.tfne["izh_current_to_ampere_scale"])
print({"spikes": spike_count, "izh_current_to_ampere_scale": scale_A})

grid = make_regular_grid((10, 10, 10), (100e-6, 100e-6, 150e-6))
# Project one time slice using the first few neurons as source locations.
source_positions = jnp.asarray(net.tfne["source_positions_m"][:5])
radii = jnp.asarray(net.tfne["source_radii_m"][:5])
currents = jnp.asarray([scale_A, -scale_A, 0.5 * scale_A, 0.0, scale_A])
q = project_sparse_currents(grid, currents, source_positions, radii)
print({"source_integral_A": float(integrate_source(grid, q)), "target_A": float(jnp.sum(currents)), "finite": bool(jnp.all(jnp.isfinite(q)))})


{'spikes': 2, 'izh_current_to_ampere_scale': 1e-12}


{'source_integral_A': 1.4999998855860786e-12, 'target_A': 1.4999999940062958e-12, 'finite': True}
